# 02 — Train one RT-DETR-X fold

Set `FOLD` in the parameters cell, run the notebook, get one checkpoint at
`runs/kfold/fold_{FOLD}_rtdetr_x/weights/best.pt`.

Design choice: one fold per kernel run. Training is hours-long and 12 GB VRAM is tight;
restarting the kernel between folds prevents state leakage and gives clean OOM-free runs.
A 5-fold driver cell is provided at the bottom, commented out, if you prefer to batch overnight.

Constants below come from `WBF_context.md` § Load-Bearing Constants. Do not change without reason.

## Reproducibility

In [1]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Parameters

In [2]:
# Choose split: set BAG to int 0..6 to train a bagging model, or leave None to train fold FOLD.
FOLD = 0       # 0..4 — used only when BAG is None
BAG  = 6       # None | 0..6

from pathlib import Path
PROJECT_ROOT = Path(r"C:/Users/Victor/Desktop/Projects/ESA_SAR_WBF")
RUNS_DIR     = PROJECT_ROOT / "runs/kfold"

if BAG is not None:
    DATA_YAML   = PROJECT_ROOT / f"bags/bag_{BAG}.yaml"
    RUN_NAME    = f"bag_{BAG}_rtdetr_x"
    SEED_OFFSET = 10 + BAG     # WBF_guidelines.md § Escalation: SEED + 10 + i, avoids fold-seed collision
    SPLIT_LABEL = f"BAG {BAG}"
else:
    DATA_YAML   = PROJECT_ROOT / f"folds/fold_{FOLD}.yaml"
    RUN_NAME    = f"fold_{FOLD}_rtdetr_x"
    SEED_OFFSET = FOLD
    SPLIT_LABEL = f"FOLD {FOLD}"

assert DATA_YAML.is_file(), f"missing {DATA_YAML} — run 01_folds.ipynb (folds) or 05_bag_splits.ipynb (bags) first"
print("split:", SPLIT_LABEL)
print("data yaml:", DATA_YAML)
print("run dir:", RUNS_DIR / RUN_NAME)
print("seed offset:", SEED_OFFSET)

split: BAG 6
data yaml: C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\bags\bag_6.yaml
run dir: C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\runs\kfold\bag_6_rtdetr_x
seed offset: 16


## Environment probe

Catch a missing GPU / wrong torch build before committing hours to a doomed run.

In [3]:
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0), "| VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
assert torch.cuda.is_available(), "no CUDA — refusing to train on CPU"

torch: 2.11.0+cu128 cuda available: True
device: NVIDIA GeForce RTX 5080 | VRAM (GB): 17.1


## Train

Per-fold seed = `SEED + FOLD` so each fold's data-order/augmentation RNG is distinct while the global seed stays fixed.
`rtdetr-x.pt` auto-downloads on first call; subsequent folds reuse the Ultralytics cache.

In [4]:
from ultralytics import YOLO

# Recipe matches Exp D's actual train_args from its checkpoint (NOT args.yaml — that file
# writes default values rather than what was actually passed, which misled us earlier).

model = YOLO("rtdetr-x.pt")

results = model.train(
    data            = str(DATA_YAML),
    epochs          = 80,
    imgsz           = 640,
    amp             = False,      # mandatory — any AMP -> GIoU NaN on RT-DETR-X
    batch           = 6,
    workers         = 4,
    device          = 0,
    project         = str(RUNS_DIR),
    name            = RUN_NAME,
    exist_ok        = False,
    seed            = SEED + SEED_OFFSET,
    deterministic   = True,
    # --- Augmentation ---
    degrees         = 0.0,
    flipud          = 0.0,
    fliplr          = 0.5,
    mosaic          = 1.0,
    close_mosaic    = 10,
    copy_paste      = 0.1,
    mixup           = 0.0,
    hsv_h           = 0.0,
    hsv_s           = 0.3,
    hsv_v           = 0.4,
    erasing         = 0.4,
    # --- Optimizer & LR schedule (matches Exp D checkpoint train_args byte-for-byte) ---
    optimizer       = "auto",     # resolves to AdamW(lr=0.002) on this 8.4.37/nc=1 setup — what Exp D used
    lr0             = 1e-5,       # ignored by auto, kept for parity with Exp D
    lrf             = 0.01,
    weight_decay    = 0.0005,
    momentum        = 0.937,
    cos_lr          = True,
    warmup_epochs   = 10,
    warmup_bias_lr  = 0.0,        # CORRECTED — Exp D's checkpoint train_args shows 0.0 (args.yaml lies)
    warmup_momentum = 0.8,
    patience        = 25,
    save            = True,
    save_period     = 10,
    val             = True,
    plots           = True,
    verbose         = True,
)

New https://pypi.org/project/ultralytics/8.4.47 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\bags\bag_6.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-x.pt, momentum=0.

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/80        12G     0.9047      1.058     0.3669         11        640: 100% ━━━━━━━━━━━━ 493/493 3.0it/s 2:43<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 6.0it/s 2.8s0.2s
                   all        200        545      0.664      0.528      0.548      0.318

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/80      11.9G     0.8071      0.535     0.1503         38        640: 0% ──────────── 0/493  0.3s

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/80      11.9G     0.5945     0.6382     0.1734          4        640: 100% ━━━━━━━━━━━━ 493/493 3.2it/s 2:34<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 6.2it/s 2.7s0.2s
                   all        200        545      0.602       0.62      0.607      0.326

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/80      12.2G     0.5672     0.6033     0.1398         41        640: 0% ──────────── 0/493  0.4s

...

      78/80        12G     0.4555     0.5725     0.1191          5        640: 100% ━━━━━━━━━━━━ 493/493 3.4it/s 2:23<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 6.3it/s 2.7s0.2s
                   all        200        545      0.693      0.667      0.702      0.419

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      79/80      12.2G     0.5308     0.6384     0.1554         22        640: 0% ──────────── 0/493  0.3s

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      79/80      12.2G     0.4538     0.5791     0.1196          4        640: 100% ━━━━━━━━━━━━ 493/493 3.5it/s 2:23<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 6.4it/s 2.6s0.2s
                   all        200        545      0.701      0.666      0.704       0.42

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      80/80      12.2G     0.4884      0.462    0.09189         17        640: 0% ──────────── 0/493  0.3s

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      80/80      12.2G     0.4525     0.5735     0.1175          3        640: 100% ━━━━━━━━━━━━ 493/493 3.5it/s 2:21<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 6.4it/s 2.6s0.2s
                   all        200        545      0.697      0.668      0.705      0.423

80 epochs completed in 3.231 hours.
Optimizer stripped from C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\runs\kfold\bag_6_rtdetr_x\weights\last.pt, 135.4MB
Optimizer stripped from C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\runs\kfold\bag_6_rtdetr_x\weights\best.pt, 135.4MB

Validating C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\runs\kfold\bag_6_rtdetr_x\weights\best.pt...
Ultralytics 8.4.37  Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
rt-detr-x summary: 378 layers, 65,469,491 parameters, 0 gradients, 222.5 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━

## Record fold summary

Append best-epoch and best val mAP@0.50:0.95 to `fold_summary.json` so `04_ensemble.ipynb`
can flag a weak-link fold (> 0.03 below median of the other 4).

In [ ]:
import json, pandas as pd

results_csv = RUNS_DIR / RUN_NAME / "results.csv"
df = pd.read_csv(results_csv)
df.columns = [c.strip() for c in df.columns]
map50_95_col = next(c for c in df.columns if "mAP50-95" in c or "map50-95" in c.lower())
best_row = df.loc[df[map50_95_col].idxmax()]
best_epoch = int(best_row["epoch"]) if "epoch" in df.columns else int(df[map50_95_col].idxmax())
best_map = float(best_row[map50_95_col])
print(f"fold {FOLD}: best epoch {best_epoch}  val mAP50-95 {best_map:.4f}")

summary_path = PROJECT_ROOT / "fold_summary.json"
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
summary[str(FOLD)] = {"best_epoch": best_epoch, "best_val_map50_95": best_map}
summary_path.write_text(json.dumps(summary, indent=2))
print("updated", summary_path)

fold 4: best epoch 21  val mAP50-95 0.4228
updated C:\Users\Victor\Desktop\Projects\ESA_SAR_WBF\fold_summary.json


## Optional: batch driver

Uncomment to run folds back-to-back from a fresh kernel. Each iteration instantiates a new
model so state doesn't leak, but they share this kernel's GPU context — fine for 4–5 sequential
runs, not ideal for many. Ask for the subprocess-per-fold variant if you want stronger isolation.

```python
# from ultralytics import YOLO
# for F in range(5):   # or range(1, 5) after fold 0 smoke-tests
#     torch.cuda.empty_cache()
#     m = YOLO("rtdetr-x.pt")
#     m.train(
#         data            = str(PROJECT_ROOT / f"folds/fold_{F}.yaml"),
#         epochs=90, imgsz=640, amp=False, batch=6, workers=4, device=0,
#         project=str(RUNS_DIR), name=f"fold_{F}_rtdetr_x", exist_ok=False,
#         seed=SEED + F, deterministic=True,
#         degrees=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, close_mosaic=10,
#         copy_paste=0.1, mixup=0.0, hsv_h=0.0, hsv_s=0.3, hsv_v=0.4,
#         optimizer="AdamW", lr0=1e-3, lrf=0.001, cos_lr=True,
#         warmup_epochs=10, warmup_bias_lr=0.0, warmup_momentum=0.937,
#         patience=70, save=True, save_period=15, val=True, plots=True, verbose=True,
#     )
#     del m
```